# 05 CatBoost Improved
Этап 11: derived features + improved CatBoost c сохранением model/meta и improved matrices.

In [1]:
import importlib
import json
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

from _shared_notebook_utils import (
    RESEARCH_CHECKPOINT_DIR,
    CATBOOST_DIR,
    save_json,
    save_pickle,
    assert_class_mapping_consistency,
)

# Load frozen baseline split from checkpoint.
baseline_dir = RESEARCH_CHECKPOINT_DIR / "baseline"
train_df = pd.read_pickle(baseline_dir / "train_df.pkl")
val_df = pd.read_pickle(baseline_dir / "val_df.pkl")
test_df = pd.read_pickle(baseline_dir / "test_df.pkl")

baseline_meta = json.loads((RESEARCH_CHECKPOINT_DIR / "baseline_meta.json").read_text(encoding="utf-8"))
target_to_id = {str(k): int(v) for k, v in baseline_meta["target_to_id"].items()}
id_to_target = {int(k): str(v) for k, v in baseline_meta["id_to_target"].items()}

for split_df in [train_df, val_df, test_df]:
    if "target_encoded" not in split_df.columns:
        split_df["target_encoded"] = split_df["target"].astype("string").map(target_to_id).astype(int)

# Feature engineering on copies of split dataframes to reduce RAM peak vs creating one huge df.
def add_derived_features(split_df):
    split_df = split_df.copy()
    history_cols = ["history_1", "history_2", "history_3"]
    split_df[history_cols] = split_df[history_cols].astype("string")
    split_df["target"] = split_df["target"].astype("string")

    h1 = split_df["history_1"]
    h2 = split_df["history_2"]
    h3 = split_df["history_3"]

    if not split_df[history_cols].isna().any(axis=None):
        split_df["n_unique_history"] = np.select(
            [
                (h1 == h2) & (h2 == h3),
                (h1 == h2) | (h1 == h3) | (h2 == h3),
            ],
            [1, 2],
            default=3,
        ).astype("int8")
    else:
        split_df["n_unique_history"] = split_df[history_cols].nunique(axis=1, dropna=True).astype("int8")

    split_df["has_fallow_in_history"] = (split_df[history_cols] == "fallow").any(axis=1).astype("int8")
    split_df["has_legume_in_history"] = (split_df[history_cols] == "legumes").any(axis=1).astype("int8")
    split_df["same_h1_h2"] = (h1 == h2).astype("int8")
    split_df["same_h2_h3"] = (h2 == h3).astype("int8")
    split_df["same_all_history"] = ((h1 == h2) & (h2 == h3)).astype("int8")
    return split_df

train_df_improved = add_derived_features(train_df)
val_df_improved = add_derived_features(val_df)
test_df_improved = add_derived_features(test_df)

improved_cat_features = ["history_1", "history_2", "history_3", "STATEFIPS", "CNTYFIPS"]
improved_num_features = [
    "CSBACRES", "INSIDE_X", "INSIDE_Y",
    "n_unique_history", "has_fallow_in_history", "has_legume_in_history",
    "same_h1_h2", "same_h2_h3", "same_all_history",
]
improved_feature_cols = improved_cat_features + improved_num_features

missing_improved_cols = [c for c in improved_feature_cols if c not in train_df_improved.columns]
if missing_improved_cols:
    raise ValueError(f"Missing improved feature columns: {missing_improved_cols}")

for split_df in [train_df_improved, val_df_improved, test_df_improved]:
    for col in improved_cat_features:
        split_df[col] = split_df[col].astype("string")

X_train_improved = train_df_improved[improved_feature_cols]
X_val_improved = val_df_improved[improved_feature_cols]
X_test_improved = test_df_improved[improved_feature_cols]

y_train_improved = train_df_improved["target_encoded"].astype(int)
y_val_improved = val_df_improved["target_encoded"].astype(int)
y_test_improved = test_df_improved["target_encoded"].astype(int)

print("Loaded baseline split for improved features:", train_df.shape, val_df.shape, test_df.shape)
print("Improved X/y shapes:")
print("X_train_improved:", X_train_improved.shape, "| y_train_improved:", y_train_improved.shape)
print("X_val_improved:", X_val_improved.shape, "| y_val_improved:", y_val_improved.shape)
print("X_test_improved:", X_test_improved.shape, "| y_test_improved:", y_test_improved.shape)
print("improved_feature_cols:", improved_feature_cols)

Loaded baseline split for improved features: (26957630, 12) (5776566, 12) (5777014, 12)
Improved X/y shapes:
X_train_improved: (26957630, 14) | y_train_improved: (26957630,)
X_val_improved: (5776566, 14) | y_val_improved: (5776566,)
X_test_improved: (5777014, 14) | y_test_improved: (5777014,)
improved_feature_cols: ['history_1', 'history_2', 'history_3', 'STATEFIPS', 'CNTYFIPS', 'CSBACRES', 'INSIDE_X', 'INSIDE_Y', 'n_unique_history', 'has_fallow_in_history', 'has_legume_in_history', 'same_h1_h2', 'same_h2_h3', 'same_all_history']


In [2]:
# 5.1 Train/evaluate/save improved CatBoost
RUN_TRAIN = False
RUN_EVAL = True
RUN_SAVE_MODEL = False
RUN_SAVE_CHECKPOINT_MATRICES = True
USE_GPU = True

improved_artifacts_dir = CATBOOST_DIR / "improved"
improved_artifacts_dir.mkdir(parents=True, exist_ok=True)
improved_model_path = improved_artifacts_dir / "model.cbm"
improved_meta_path = improved_artifacts_dir / "meta.json"
improved_train_dir = improved_artifacts_dir / "catboost_info"
improved_train_dir.mkdir(parents=True, exist_ok=True)

catboost_mod = importlib.import_module("catboost")
CatBoostClassifier = catboost_mod.CatBoostClassifier

if RUN_TRAIN:
    catboost_params_improved = {
        "loss_function": "MultiClass",
        "eval_metric": "TotalF1:average=Macro",
        "iterations": 1000,
        "learning_rate": 0.05,
        "depth": 6,
        "l2_leaf_reg": 3.0,
        "random_seed": 42,
        "early_stopping_rounds": 50,
        "task_type": "GPU" if USE_GPU else "CPU",
        "train_dir": improved_train_dir.as_posix(),
        "verbose": 100,
    }
    if USE_GPU:
        catboost_params_improved["devices"] = "0"
        catboost_params_improved["gpu_ram_part"] = 0.80
        catboost_params_improved["max_ctr_complexity"] = 1

    improved_cat_feature_indices = [improved_feature_cols.index(col) for col in improved_cat_features]
    improved_cat_model = CatBoostClassifier(**catboost_params_improved)
    improved_cat_model.fit(
        X_train_improved,
        y_train_improved,
        cat_features=improved_cat_feature_indices,
        eval_set=(X_val_improved, y_val_improved),
        use_best_model=True,
    )

    print("Improved model training completed.")
    print("Best iteration:", improved_cat_model.get_best_iteration())
    print("Best validation score:", improved_cat_model.get_best_score())
elif improved_model_path.exists():
    improved_cat_model = CatBoostClassifier()
    improved_cat_model.load_model(improved_model_path.as_posix())
    print("Loaded existing improved model:", improved_model_path.resolve())
else:
    raise ValueError("No improved model available. Set RUN_TRAIN=True or provide artifacts/catboost/improved/model.cbm")

def flatten_catboost_preds(pred):
    return np.asarray(pred).reshape(-1).astype(int)

if RUN_EVAL:
    val_pred_improved = flatten_catboost_preds(improved_cat_model.predict(X_val_improved))
    test_pred_improved = flatten_catboost_preds(improved_cat_model.predict(X_test_improved))

    improved_metrics_df = pd.DataFrame(
        [
            {
                "split": "validation",
                "accuracy": float(accuracy_score(y_val_improved, val_pred_improved)),
                "macro_f1": float(f1_score(y_val_improved, val_pred_improved, average="macro", zero_division=0)),
                "weighted_f1": float(f1_score(y_val_improved, val_pred_improved, average="weighted", zero_division=0)),
            },
            {
                "split": "test",
                "accuracy": float(accuracy_score(y_test_improved, test_pred_improved)),
                "macro_f1": float(f1_score(y_test_improved, test_pred_improved, average="macro", zero_division=0)),
                "weighted_f1": float(f1_score(y_test_improved, test_pred_improved, average="weighted", zero_division=0)),
            },
        ]
    )
    print("Improved model metrics:")
    display(improved_metrics_df)

    labels_sorted = sorted(id_to_target.keys())
    target_names = [id_to_target[i] for i in labels_sorted]

    print("Classification report (validation):")
    print(classification_report(y_val_improved, val_pred_improved, labels=labels_sorted, target_names=target_names, zero_division=0))
    print("Classification report (test):")
    print(classification_report(y_test_improved, test_pred_improved, labels=labels_sorted, target_names=target_names, zero_division=0))

    cm_test_improved_df = pd.DataFrame(
        confusion_matrix(y_test_improved, test_pred_improved, labels=labels_sorted),
        index=[f"true_{id_to_target[i]}" for i in labels_sorted],
        columns=[f"pred_{id_to_target[i]}" for i in labels_sorted],
    )
    print("Confusion matrix (test):")
    display(cm_test_improved_df)

if RUN_SAVE_MODEL:
    improved_cat_model.save_model(improved_model_path.as_posix())
    improved_meta = {
        "feature_cols": improved_feature_cols,
        "cat_features": improved_cat_features,
        "num_features": improved_num_features,
        "target_column": "target",
        "encoded_target_column": "target_encoded",
        "target_to_id": {str(k): int(v) for k, v in target_to_id.items()},
        "id_to_target": {str(int(k)): str(v) for k, v in id_to_target.items()},
        "split_source": "research_checkpoint/baseline",
    }
    save_json(improved_meta, improved_meta_path)
    print("Improved model saved to:", improved_model_path.resolve())
    print("Improved metadata saved to:", improved_meta_path.resolve())

if RUN_SAVE_CHECKPOINT_MATRICES:
    improved_ckpt_dir = RESEARCH_CHECKPOINT_DIR / "improved"
    improved_ckpt_dir.mkdir(parents=True, exist_ok=True)
    save_pickle(X_train_improved, improved_ckpt_dir / "X_train_improved.pkl")
    save_pickle(X_val_improved, improved_ckpt_dir / "X_val_improved.pkl")
    save_pickle(X_test_improved, improved_ckpt_dir / "X_test_improved.pkl")
    save_pickle(y_train_improved, improved_ckpt_dir / "y_train_improved.pkl")
    save_pickle(y_val_improved, improved_ckpt_dir / "y_val_improved.pkl")
    save_pickle(y_test_improved, improved_ckpt_dir / "y_test_improved.pkl")

    improved_meta_ckpt = {
        "improved_feature_cols": improved_feature_cols,
        "improved_cat_features": improved_cat_features,
        "improved_num_features": improved_num_features,
        "improved_target_col": "target_encoded",
        "target_to_id_improved": {str(k): int(v) for k, v in target_to_id.items()},
        "id_to_target_improved": {str(int(k)): str(v) for k, v in id_to_target.items()},
    }

    # Ensure mapping consistency between baseline and improved artifacts.
    assert_class_mapping_consistency(target_to_id, improved_meta_ckpt["target_to_id_improved"])
    save_json(improved_meta_ckpt, RESEARCH_CHECKPOINT_DIR / "improved_meta.json")
    print("Saved improved checkpoint matrices and improved_meta.json")

Loaded existing improved model: C:\Users\Dmitry\code-projects\diploma-crop-rotation\artifacts\catboost\improved\model.cbm
Improved model metrics:


,split,accuracy,macro_f1,weighted_f1
0,validation,0.699525,0.570025,0.690367
1,test,0.699235,0.569473,0.689991


Classification report (validation):
               precision    recall  f1-score   support

         corn       0.69      0.72      0.71   1763620
       cotton       0.72      0.77      0.74    278319
       fallow       0.59      0.41      0.49    270527
   forage_hay       0.81      0.85      0.83    561759
      legumes       0.50      0.14      0.22     86921
other_cereals       0.53      0.24      0.33    150744
      sorghum       0.50      0.31      0.38    173768
     soybeans       0.72      0.76      0.74   1645760
        wheat       0.66      0.73      0.69    845148

     accuracy                           0.70   5776566
    macro avg       0.63      0.55      0.57   5776566
 weighted avg       0.69      0.70      0.69   5776566

Classification report (test):
               precision    recall  f1-score   support

         corn       0.69      0.72      0.71   1762382
       cotton       0.72      0.77      0.74    277223
       fallow       0.59      0.41      0.48    27

,pred_corn,pred_cotton,pred_fallow,pred_forage_hay,pred_legumes,pred_other_cereals,pred_sorghum,pred_soybeans,pred_wheat
true_corn,1265050,27089,17862,58708,2728,5957,12597,317533,54858
true_cotton,15968,213073,2314,1625,24,384,6904,17725,19206
true_fallow,34038,5554,111526,4196,931,4410,11647,45190,53194
true_forage_hay,43041,1641,2438,476657,527,5715,701,14048,15624
true_legumes,12706,303,1903,2447,12397,1956,205,15571,40176
true_other_cereals,29266,2442,9500,12569,511,36612,2899,11115,46446
true_sorghum,29537,13967,18152,2331,39,2251,53058,9467,44775
true_soybeans,305956,14489,4872,12703,3762,884,3110,1251036,48809
true_wheat,87947,17635,20858,17674,3853,11152,16572,52339,620079


Saved improved checkpoint matrices and improved_meta.json
